# Prepare OMERO Data for BiaPy training

Train AI models using annotation tables from OMERO with automated data preparation.

## 1. Setup

In [ ]:
# Import the package with training convenience functions
from omero_annotate_ai import (
    create_omero_connection_widget,
    create_training_data_widget,
    prepare_training_data_from_table,

)

# Additional imports
import datetime
from pathlib import Path
import torch

print(f"Check if GPU is available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. OMERO Connection

In [ ]:
# Create and display OMERO connection widget
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
# Get the OMERO connection
conn = conn_widget.get_connection()

if conn is None:
    raise ConnectionError("No OMERO connection established.")

print(f"Connected to OMERO as: {conn.getUser().getName()}")

## 3. Training Data Selection

In [ ]:
# Create training data selection widget
training_widget = create_training_data_widget(connection=conn)
training_widget.display()

In [ ]:
# Get selected training table
selected_table_id = training_widget.get_selected_table_id()
selected_table_info = training_widget.get_selected_table_info()

if selected_table_id:
    print(f"Selected training table:")
    print(f"  Table ID: {selected_table_id}")
    print(f"  Table Name: {selected_table_info.get('name', 'Unknown')}")
    print(f"  Created: {selected_table_info.get('created', 'Unknown')}")
else:
    raise ValueError("No training table selected. Please select a table above.")

## 4. Setup Training Directory

In [ ]:
# Create output directory for training
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
home_dir = Path.home()
models_dir = home_dir / "biapy_data"
models_dir.mkdir(exist_ok=True)

folder_name = f"training_data-{timestamp}"
output_directory = models_dir / folder_name
output_directory.mkdir(exist_ok=True)

print(f"Training output directory: {output_directory}")

## 5. Automated Data Preparation

Use the automated data preparation function to download and organize training data.

In [ ]:
# Run automated data preparation
try:
    training_result = prepare_training_data_from_table(
        conn=conn,
        table_id=selected_table_id,
        training_name=selected_table_info.get('name', f"training_table_{selected_table_id}"),
        output_dir=output_directory,
        clean_existing=True,
        verbose=False
    )
    
    print("\nTraining data preparation completed successfully!")
    print(f"\nDataset statistics:")
    for key, value in training_result['stats'].items():
        print(f"  {key}: {value}")
    
    training_input_dir = training_result['training_input']
    training_label_dir = training_result['training_label']
    val_input_dir = training_result['val_input']
    val_label_dir = training_result['val_label']
    
    print(f"\nDirectory structure created:")
    print(f"  Training images: {training_input_dir}")
    print(f"  Training labels: {training_label_dir}")
    print(f"  Validation images: {val_input_dir}")
    print(f"  Validation labels: {val_label_dir}")
    
    (output_directory / 'output').mkdir(parents=True, exist_ok=True)
    print(f"\nPaths to use for BiaPy training:")
    print(f"  Data folder: {output_directory}")
    print(f"  Output folder: {output_directory / 'output'}")
    
except Exception as e:
    print(f"Error during data preparation: {e}")
    raise

## 6. Export the trained model to the BioImage Model Zoo

After training with BiaPy (outside this notebook), package the model for the BioImage
Model Zoo, reusing the metadata captured in your `AnnotationConfig`. Run this section in
the environment where you trained with BiaPy.

In [ ]:
# Reuse the metadata from the AnnotationConfig you created during annotation. It was
# attached to the OMERO container (Dataset/Plate) you annotated -- set that container below.
from omero_annotate_ai.omero import download_annotation_config_from_omero
from omero_annotate_ai.core.modelzoo_metadata import (
    config_to_bmz_metadata,
    to_biapy_bmz_cfg,
)

annotated_container_type = "Dataset"   # or "Plate"
annotated_container_id = 0             # <-- set to the OMERO container ID you annotated
config = download_annotation_config_from_omero(
    conn, annotated_container_type, annotated_container_id
)
# Or: from omero_annotate_ai.core.annotation_config import load_config_from_yaml

bmz_cfg = to_biapy_bmz_cfg(config_to_bmz_metadata(config))
bmz_cfg

In [ ]:
bmz_folder = str(output_directory / "bmz_model")
try:
    from biapy import BiaPy  # noqa: F401
except ImportError:
    print(
        "BiaPy is not installed in this environment. Run this section where you trained "
        "with BiaPy (pip install biapy); the bmz_cfg above is ready to use."
    )
else:
    # `biapy` is your trained BiaPy job: biapy = BiaPy(...); biapy.run_job()
    biapy.export_model_to_bmz(bmz_folder, bmz_cfg=bmz_cfg)
    print(f"Exported BMZ model to: {bmz_folder}")